In [7]:
import pandas as pd

# Load baseline results
baseline_df = pd.read_csv('baseline_results.csv')

# Load RL results and compute filter only eval episodes
df = pd.read_csv('rl_evaders_results.csv')
eval_df = df[df['phase'] == 'eval']

# Compute overall outcomes from eval episodes 
evader_results = (
    eval_df
    .groupby(['pursuer_baseline', 'n_pursuers', 'n_evaders'])
    .agg(
        episodes=('turns', 'count'),
        evader_wins=('evader_win', 'sum'),
        avg_turns=('turns', 'mean'),
        avg_flag_transfers=('flag_transfers', 'mean')
    )
    .reset_index()
)

# Match the columns with baseline dataset
evader_results['evader_win_rate'] = evader_results['evader_wins'] / evader_results['episodes']
evader_results['evader_baseline'] = 4
evader_results['pursuer_roles'] = None
evader_results['evader_roles'] = None

evader_results = evader_results[baseline_df.columns]

# Add RL rows into the correct pursuer_baseline section
sections = []
for baseline in sorted(baseline_df['pursuer_baseline'].unique()):
    baseline_section = baseline_df[baseline_df['pursuer_baseline'] == baseline]
    rl_section = evader_results[evader_results['pursuer_baseline'] == baseline]
    sections.append(pd.concat([baseline_section, rl_section], ignore_index=True))

baseline_evader_results = pd.concat(sections, ignore_index=True)

baseline_evader_results

,pursuer_baseline,evader_baseline,n_pursuers,n_evaders,pursuer_roles,evader_roles,episodes,evader_wins,evader_win_rate,avg_turns,avg_flag_transfers
0,1,1,4,4,"chase,chase,chase,chase","escape,support,support,support",20,0,0.0,11.0,1.0
1,1,1,4,6,"chase,chase,chase,chase","escape,support,support,support,support,support",20,0,0.0,11.0,1.0
2,1,1,4,8,"chase,chase,chase,chase","escape,support,support,support,support,support...",20,20,1.0,200.0,65.0
3,1,1,6,4,"chase,chase,chase,chase,chase,chase","escape,support,support,support",20,0,0.0,11.0,1.0
4,1,1,6,6,"chase,chase,chase,chase,chase,chase","escape,support,support,support,support,support",20,0,0.0,11.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
103,3,4,6,6,None,None,20,0,0.0,14.0,6.0
104,3,4,6,8,None,None,20,20,1.0,200.0,200.0
105,3,4,8,4,None,None,20,0,0.0,5.0,5.0
106,3,4,8,6,None,None,20,0,0.0,5.0,5.0


In [8]:
# Load RL results and compute filter only eval episodes
df = pd.read_csv('rl_pursuers_results.csv')
eval_df = df[df['phase'] == 'eval']

# Compute overall outcomes from eval episodes 
pursuer_results = (
    eval_df
    .groupby(['evader_baseline', 'n_pursuers', 'n_evaders'])
    .agg(
        episodes=('turns', 'count'),
        evader_wins=('evader_win', 'sum'),
        avg_turns=('turns', 'mean'),
        avg_flag_transfers=('flag_transfers', 'mean')
    )
    .reset_index()
)

# Match columns with baseline dataset
pursuer_results['evader_win_rate'] = pursuer_results['evader_wins'] / pursuer_results['episodes']
pursuer_results['pursuer_baseline'] = 4
pursuer_results['pursuer_roles'] = None
pursuer_results['evader_roles'] = None

pursuer_results = pursuer_results[baseline_df.columns]
pursuer_results

,pursuer_baseline,evader_baseline,n_pursuers,n_evaders,pursuer_roles,evader_roles,episodes,evader_wins,evader_win_rate,avg_turns,avg_flag_transfers
0,4,1,4,4,None,None,20,0,0.0,6.0,0.0
1,4,1,4,6,None,None,20,0,0.0,6.0,0.0
2,4,1,4,8,None,None,20,0,0.0,22.0,3.0
3,4,1,6,4,None,None,20,0,0.0,5.0,0.0
4,4,1,6,6,None,None,20,0,0.0,5.0,0.0
5,4,1,6,8,None,None,20,0,0.0,5.0,0.0
6,4,1,8,4,None,None,20,0,0.0,6.0,0.0
7,4,1,8,6,None,None,20,0,0.0,5.0,0.0
8,4,1,8,8,None,None,20,0,0.0,5.0,0.0
9,4,2,4,4,None,None,20,0,0.0,1.0,1.0


In [9]:
# Add RL evader results onto the end of existing results set
final_results = pd.concat([baseline_evader_results, pursuer_results], ignore_index=True)
final_results

,pursuer_baseline,evader_baseline,n_pursuers,n_evaders,pursuer_roles,evader_roles,episodes,evader_wins,evader_win_rate,avg_turns,avg_flag_transfers
0,1,1,4,4,"chase,chase,chase,chase","escape,support,support,support",20,0,0.0,11.0,1.0
1,1,1,4,6,"chase,chase,chase,chase","escape,support,support,support,support,support",20,0,0.0,11.0,1.0
2,1,1,4,8,"chase,chase,chase,chase","escape,support,support,support,support,support...",20,20,1.0,200.0,65.0
3,1,1,6,4,"chase,chase,chase,chase,chase,chase","escape,support,support,support",20,0,0.0,11.0,1.0
4,1,1,6,6,"chase,chase,chase,chase,chase,chase","escape,support,support,support,support,support",20,0,0.0,11.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
130,4,3,6,6,None,None,20,0,0.0,5.0,0.0
131,4,3,6,8,None,None,20,0,0.0,5.0,0.0
132,4,3,8,4,None,None,20,0,0.0,5.0,0.0
133,4,3,8,6,None,None,20,0,0.0,5.0,0.0


In [10]:
# Save overall dataset
final_results.to_csv('final_results.csv', index=False) 